# Notebook 3 : Données boursières 💸

In [ ]:
# Décommenter la ligne suivante pour installer les dépendances
# %pip install jupyter_bokeh nbconvert panel watchfiles

In [ ]:
import pandas as pd
import panel as pn
import plotly.express as px
import plotly.graph_objects as go

pn.extension("plotly")

Nous proposons ici de mettre en place une application web pour visualiser des données boursières ainsi que des indicateurs statistiques relatifs aux actions suivantes pour la période du 2 janvier 2020 au 5 juin 2024 :

- `CAC` CAC 40 (*Cotation Assistée en Continu*),
- `DJIA` Dow Jones Industrial Average,
- `SPX` S&P 500 (*Standard and Poor's 500*).

## Préliminaires

1. Écrire une fonction `get_data` sans argument qui retourne les données contenues dans le fichier `data/bourse.csv` sous la forme d'un DataFrame Pandas. La variable `Date` contient des chaînes de caractères et elle devra être convertie en date à l'aide de `pd.to_datetime` (voir [la documentation](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.to_datetime.html), en particulier pour le paramètre `format`) pour faciliter l'utilisation du DataFrame dans la suite. Cette fontion a vocation à être appelée plusieurs fois avec le même résultat attendu, elle pourra donc être décorée avec `pn.cache`.

In [ ]:
@pn.cache
def get_data():
    data_path = "data/bourse.csv"
    print(f"Charge les données depuis {data_path}")
    data = pd.read_csv(data_path)

    # Conversion de la variable Date
    # (%d pour le jour, %m pour le mois et %y pour l'année sans les centaines)
    data.Date = pd.to_datetime(data.Date, format="%m/%d/%y").dt.date

    return data.sort_values(by="Date")

2. Utiliser la fonction `line` de Plotly (voir [la documentation](https://plotly.com/python-api-reference/generated/plotly.express.line)) pour afficher la série temporelle des prix à la fermeture `Close` de l'action `CAC` sur toute la période du jeu de données. Ajouter des noms aux axes grâce aux méthodes `update_xaxes` et `update_yaxes` (voir [la documentation](https://plotly.com/python/axes/#set-axis-title-text-with-graph-objects)).

In [ ]:
bourse = get_data()
cac = bourse[bourse.Stock == "CAC"]

fig = px.line(cac, x="Date", y="Close")
fig.update_xaxes(title_text="Date")
fig.update_yaxes(title_text="Prix")

3. Enrichir le graphique précédent en ajoutant :

- un titre principal pour la figure,
- des points pour indiquer les valeurs minimales `Low` et maximales `High` à chaque date avec la méthode `add_trace` et la fonction `Scatter` du module Plotly `graph_objects` (alias `go`) qui fournit des fonctionnalités graphiques plus génériques (voir les exemples de [la documentation](https://plotly.com/python/line-charts/#line-plot-with-goscatter)),
- un survol des valeurs amélioré avec l'option `hovermode="x unified"` de la méthode `update_layout` (voir [la documentation](https://plotly.com/python/reference/layout/)).

In [ ]:
# Figure précédente avec un titre
fig = px.line(cac, x="Date", y="Close", title="Action CAC")
fig.update_xaxes(title_text="Date")
fig.update_yaxes(title_text="Prix")

# Ajout des valeurs Low et High avec un nom pour la légende
fig.add_trace(go.Scatter(x=cac.Date, y=cac.Low, mode="markers", name="Low"))
fig.add_trace(go.Scatter(x=cac.Date, y=cac.High, mode="markers", name="High"))

# Survol des valeurs amélioré
fig.update_layout(hovermode="x unified")

La [moyenne mobile](https://fr.wikipedia.org/wiki/Moyenne_mobile) est un outil statistique utilisé pour l'analyse et la visualisation des séries temporelles. Elle permet de lisser les fluctuations et de faire ressortir une tendance en calculant la moyenne d'un nombre fixé d'observations (appelé *fenêtre* ou *window*) précédent un instant donné.

Formellement, si nous disposons d'une série de valeurs $(x_k)_{k \geq 0}$, la moyenne mobile de fenêtre $W > 0$ à l'instant $n$ est donnée par
$$\bar{x}_n = \frac{1}{W} \sum_{k=1}^W x_{n-k+1}$$

4. Utiliser la méthode `rolling` (voir [la documentation](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.rolling.html)) sur la Series Pandas des valeurs de fermeture `Close` pour calculer la moyenne mobile de fenêtre 20. Afficher cette Series comme dans la question 2 avec sa moyenne mobile dans un même graphique à l'aide de la fonction `Scatter` du module Plotly `graph_objects`.

In [ ]:
# Moyenne mobile de fenêtre 20
mob = cac.Close.rolling(window=20).mean()

# Figure précédente
fig = px.line(cac, x="Date", y="Close", title="Action CAC avec moyenne mobile")
fig.update_xaxes(title_text="Date")
fig.update_yaxes(title_text="Prix")

# Ajout de la moyenne mobile
fig.add_trace(
    go.Scatter(x=cac.Date, y=mob, mode="lines", name="Moyenne mobile")
)

## Quelques widgets

5. Créer un sélecteur `stock` de type `Select` (voir [la documentation](https://panel.holoviz.org/reference/widgets/Select.html)) pour sélectionner une des actions disponibles dans le jeu de données. Quel est le contenu et le type de l'attribut `value` de cet objet ?

In [ ]:
stock = pn.widgets.Select(
    name="Action", options=sorted(bourse.Stock.unique().tolist())
)

print(type(stock.value)) # Type str
print(stock.value) # Chaîne de caractères du nom de l'action choisie

stock

6. Créer un sélecteur de plage de dates `date_range` de type `DateRangePicker` (voir [la documentation](https://panel.holoviz.org/reference/widgets/DateRangePicker.html)) pour sélectionner une période à afficher comprise dans le jeu de données. Par défaut, la période maximale sera sélectionnée. Quel est le contenu et le type de l'attribut `value` de cet objet ?

In [ ]:
date_range = pn.widgets.DateRangePicker(
    name="Période",
    start=bourse.Date.min(),
    end=bourse.Date.max(),
    value=(bourse.Date.min(), bourse.Date.max()),
)

print(type(date_range.value)) # Type tuple
print(date_range.value) # Paire des dates choisies

date_range

7. Créer un widget `title` de type `TextInput` (voir [la documentation](https://panel.holoviz.org/reference/widgets/TextInput.html)) pour donner un titre au graphique. Quel est le contenu et le type de l'attribut `value` de cet objet ?

In [ ]:
title = pn.widgets.TextInput(name="Titre", value="Titre du graphique")

print(type(title.value)) # Type str
print(title.value) # Chaîne de caractères du titre

title

8. Créer deux interrupteurs `amplitude` et `moy_mobile` de type `Switch` pour activer/désactiver respectivement l'affichage des valeurs extrêmes à une date donnée et la moyenne mobile. Créer aussi une glissière `window` de type `IntSlider` (voir [la documentation](https://panel.holoviz.org/reference/widgets/IntSlider.html)) pour sélectionner la fenêtre de la moyenne mobile entre `1` et `50` avec une valeur par défaut de `20`. Quels sont les contenus et les types des attributs `value` de ces objets ?

In [ ]:
# Interrupteurs (l'option align règle la position du widget dans l'application)
amplitude = pn.widgets.Switch(name="Amplitude", align="center")
moy_mobile = pn.widgets.Switch(name="Moyenne mobile", align="center")

print(type(moy_mobile.value)) # Type bool
print(moy_mobile.value) # Valeur booléenne de l'interrupteur

window = pn.widgets.IntSlider(name="Fenêtre", start=1, end=50, value=20)

print(type(window.value)) # Type int
print(window.value) # Valeur de la fenêtre choisie

pn.Column(
    pn.Row(amplitude, pn.pane.Markdown(amplitude.name)),
    pn.Row(moy_mobile, pn.pane.Markdown(moy_mobile.name)),
    window
)

## Fonction graphique

9. Écrire une fonction `get_plot` qui retourne l'objet graphique Plotly du tracé de la valeur de l'action `stock` sur la période `date_range` avec :

- le titre `title`,
- les valeurs extrêmes à chaque date si `amplitude` est `True`,
- la moyenne mobile de fenêtre `window` si `moy_mobile` est `True`,
- le survol amélioré des valeur avec `hovermode="x unified"`.

In [ ]:
def get_plot(stock, date_range, title, amplitude, moy_mobile, window):
    # Filtre des données
    bourse = get_data()
    # Action stock
    df = bourse[bourse.Stock == stock].copy()
    # Moyenne mobile
    df["mobile"] = df.Close.rolling(window=window).mean()
    # Période sélectionnée
    df = df[(date_range[0] <= df.Date) & (df.Date <= date_range[1])]

    # Tracé des valeurs de fermeture de l'action avec un titre
    fig = px.line(df, x="Date", y="Close", title=title)
    fig.update_xaxes(title_text="Date")
    fig.update_yaxes(title_text="Prix")

    # Ajout des valeurs extrêmes si amplitude vaut True
    if amplitude:
        fig.add_trace(
            go.Scatter(x=df.Date, y=df.Low, mode="markers", name="Low")
        )
        fig.add_trace(
            go.Scatter(x=df.Date, y=df.High, mode="markers", name="High")
        )
    
    # Ajout de la moyenne mobile si moy_mobile vaut True
    if moy_mobile:
        fig.add_trace(
            go.Scatter(
                x=df.Date,
                y=df.mobile,
                mode="lines",
                name="Moyenne mobile"
            )
        )
        
    # Survol des valeurs amélioré
    fig.update_layout(hovermode="x unified")

    return fig

## Application web

Nous pouvons maintenant mettre en place une application web qui pourra être démarrée avec la commande suivante (l'option `--allow-websocket-origin` n'est nécessaire que dans Onyxia) :
```{bash}
panel serve --autoreload --show --allow-websocket-origin=$(echo $VSCODE_PROXY_URI | cut -d '/' -f 3) notebooks/03_bourse.ipynb
```

Les questions suivantes ont pour objet d'enrichir l'application au fur et à mesure. Il ne faut donc pas recréer une nouvelle application pour chaque question mais faire évoluer le code étape par étape. Il peut être utile d'ajouter de nouvelles cellules de code si besoin.

10. Mettre en forme une application à l'aide du modèle `FastListTemplate` (voir [la documentation](https://panel.holoviz.org/reference/templates/FastListTemplate.html)) avec les widgets `stock`, `date_range`, `title`, `amplitude`, `moy_mobile` et `window` dans la barre latérale et un *pane* contenant le résultat de la fonction `get_plot` liée aux widgets précédents dans la zone principale.

11. Utiliser le paramètre `sizing_mode` du *pane* graphique pour adapter sa taille.

In [ ]:
# Fonction graphique liée aux widget
plot = pn.bind(
    get_plot,
    stock=stock,
    date_range=date_range,
    title=title,
    amplitude=amplitude,
    moy_mobile=moy_mobile,
    window=window,
)

# Modèle de l'application
pn.template.FastListTemplate(
    title="Données boursières 💸",
    sidebar=[
        stock,
        date_range,
        title,
        pn.Column(
            pn.Row(amplitude, pn.pane.Markdown(amplitude.name)),
            pn.Row(moy_mobile, pn.pane.Markdown(moy_mobile.name)),
            window
        )
    ],
    main=[
        pn.pane.Plotly(
            plot,
            #sizing_mode="stretch_both" # Question 11
        ),
    ],
).servable()